# Blood Atlas diagnostics

This notebook is intentionally **diagnostic rather than part of the primary benchmark**. It addresses the questions raised by the unusually low Blood Atlas macro-F1 and the very small random-cell vs donor-held-out gap.

The order matters:

1. audit source-expression / metadata alignment before making biological claims;
2. characterize which classes fail and whether broad lineage markers make sense;
3. quantify donor-specific cell-type composition and the effect of the existing donor × cell-type Stage-0 cap;
4. probe donor-count sensitivity with repeated 20-donor cohorts and nested donor ablation;
5. use label-permutation controls to separate true biological separability from donor-prior leakage;
6. coarsen the label hierarchy to test whether T-cell subtype granularity explains the performance floor;
7. optionally create a donor-only, composition-preserving Stage-0 cohort and compare it with the primary capped cohort.

The primary Blood benchmark is **not modified** by this notebook. Expensive sensitivity outputs are written under `results/blood_atlas_diagnostics/` and are re-used when present.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
from IPython.display import display

# Find repository root whether Jupyter was launched from repo root or notebooks/.
HERE = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in [HERE, *HERE.parents] if (p / "src/scrna_benchmark").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate repository root containing src/scrna_benchmark")

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from scrna_benchmark.diagnostics import (
    audit_metadata_alignment,
    composition_summary_row,
    donor_celltype_counts,
    donor_celltype_proportions,
    marker_group_means,
    summarize_donor_composition,
    zscore_columns,
)
from scrna_benchmark.plotting import (
    load_per_class_f1_matrix,
    plot_confusion_matrix_from_predictions,
    plot_per_class_f1_heatmap,
)

PRIMARY_PATH = REPO_ROOT / "data/blood_atlas/blood_atlas_benchmark_ready.h5ad"
RESULTS_DIR = REPO_ROOT / "results/blood_atlas_no_batch"
DONOR_CV_DIR = RESULTS_DIR / "donor_cv"
DIAG_RESULTS = REPO_ROOT / "results/blood_atlas_diagnostics"
DIAG_FIGURES = REPO_ROOT / "figures/blood_atlas_diagnostics"
DIAG_RESULTS.mkdir(parents=True, exist_ok=True)
DIAG_FIGURES.mkdir(parents=True, exist_ok=True)

CELLTYPE_COL = "cell_type"
DONOR_COL = "donor_id"
REP_ORDER = ["hvg", "pca", "harmony"]

adata = sc.read_h5ad(PRIMARY_PATH)
print(adata)
print(f"cells={adata.n_obs:,}; genes/HVGs={adata.n_vars:,}; donors={adata.obs[DONOR_COL].nunique()}; cell types={adata.obs[CELLTYPE_COL].nunique()}")

AnnData object with n_obs × n_vars = 108682 × 1000
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_HTO', 'nFeature_HTO', 'percent.mt', 'percent.ribo', 'log2_nCount', 'log2_nFeature', 'log2_mt', 'Donor_id', 'Age_group', 'Sex', 'Age', 'Tube_id', 'Batch', 'File_name', 'Cluster_names', 'Cluster_numbers', 'donor_id', 'cell_type', 'batch', 'sample_id', 'age', 'age_group', 'sex'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'hvg', 'pca'
    obsm: 'X_harmony', 'X_pca'
    varm: 'PCs'
    layers: None (.X)
cells=108,682; genes/HVGs=1,000; donors=166; cell types=7


## Configuration for optional source audit and compute-heavy probes

The current benchmark-ready object is enough for confusion matrices, marker checks, post-cap composition, 20-donor cohorts, permutation controls, and label coarsening.

The **pre-cap composition** and **composition-preserving alternative cohort** additionally require the original Blood Atlas metadata / H5AD. Set the paths below or provide the environment variables. Nothing is silently rebuilt if those files are absent.

In [2]:
# Override these with environment variables if your source files live elsewhere.
FULL_H5AD = Path(os.environ.get(
    "BLOOD_ATLAS_FULL_H5AD",
    "/fastscratch/myscratch/xchen5/all_pbmcs/all_pbmcs_rna.h5ad",
)).expanduser()
FULL_METADATA = Path(os.environ.get(
    "BLOOD_ATLAS_METADATA_CSV",
    "/fastscratch/myscratch/xchen5/all_pbmcs/all_pbmcs_metadata.csv",
)).expanduser()
METADATA_ID_COL = os.environ.get("BLOOD_ATLAS_METADATA_ID_COL", "Unnamed: 0")

# Compute-heavy analyses are opt-in so reopening the notebook does not rerun them.
RUN_DONOR20 = False
RUN_DONOR20_ABLATION = False
RUN_PERMUTATIONS = False
RUN_COARSE_LABELS = False
RUN_COMPOSITION_STAGE0 = False
RUN_COMPOSITION_PREPROCESSING = False
RUN_COMPOSITION_EVAL = False
RUN_COMPOSITION_PERMUTATION = False

print("Full H5AD exists:", FULL_H5AD.exists(), FULL_H5AD)
print("Metadata CSV exists:", FULL_METADATA.exists(), FULL_METADATA)

Full H5AD exists: True /fastscratch/myscratch/xchen5/all_pbmcs/all_pbmcs_rna.h5ad
Metadata CSV exists: True /fastscratch/myscratch/xchen5/all_pbmcs/all_pbmcs_metadata.csv


# 1. Priority-0 audit: are source metadata and expression rows aligned?

The historical preparation notebook assigned external metadata after checking row count, so we should explicitly distinguish:

- **same cell-ID set**, which makes barcode-based reindexing safe;
- **same row order**, which is required for positional assignment.

Do not interpret the Blood Atlas biology from the full source until this audit is resolved. The primary 108k checkpoint remains loadable regardless.

In [3]:
ALIGNMENT_AUDIT_PATH = DIAG_RESULTS / "alignment/full_source_metadata_alignment_audit.csv"
alignment_audit = None

if FULL_H5AD.exists() and FULL_METADATA.exists():
    metadata = pd.read_csv(FULL_METADATA)
    full_backed = sc.read_h5ad(FULL_H5AD, backed="r")
    try:
        alignment_audit = audit_metadata_alignment(
            full_backed.obs_names,
            metadata,
            METADATA_ID_COL,
        )
    finally:
        try:
            full_backed.file.close()
        except Exception:
            pass
    ALIGNMENT_AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
    alignment_audit.to_csv(ALIGNMENT_AUDIT_PATH, index=False)
    display(alignment_audit)

    row = alignment_audit.iloc[0]
    if row["safe_for_row_order_assignment"]:
        print("PASS: metadata IDs exactly match expression rows in the current order.")
    elif row["safe_for_id_reindex"]:
        print("IMPORTANT: the ID sets match, but row order does not. Reindex by barcode; do not assign by position.")
    else:
        print("STOP for full-source analyses: metadata/expression ID sets are not safely alignable yet.")
else:
    print("Source files not found; alignment audit skipped. Set FULL_H5AD/FULL_METADATA before creating the composition-preserving cohort.")

,n_obs,n_metadata_rows,obs_names_unique,metadata_ids_unique,same_length,set_equal,n_intersection,intersection_fraction_obs,intersection_fraction_metadata,jaccard_id_overlap,order_match_fraction,safe_for_row_order_assignment,safe_for_id_reindex
0,1916367,1916367,True,True,True,True,1916367,1.0,1.0,1.0,0.0,False,True


IMPORTANT: the ID sets match, but row order does not. Reindex by barcode; do not assign by position.


# 2. Current Blood benchmark: class balance and donor support

Overall class balance can look quite uniform after the donor × cell-type Stage-0 cap even when **within-donor** composition differs. We therefore show both global counts and donor coverage.

In [ ]:
class_summary = pd.DataFrame({
    "n_cells": adata.obs[CELLTYPE_COL].astype(str).value_counts(),
    "n_donors": adata.obs.groupby(CELLTYPE_COL, observed=True)[DONOR_COL].nunique(),
}).sort_values("n_cells", ascending=False)
class_summary["fraction_cells"] = class_summary["n_cells"] / adata.n_obs
class_summary.to_csv(DIAG_RESULTS / "current_class_summary.csv")
display(class_summary)

# 3. Which classes actually fail? Pooled donor-CV confusion matrices

The primary benchmark already writes pooled donor-CV predictions. No model rerun is needed here. Row-normalized confusion matrices show whether the low macro-F1 is concentrated among neighboring T-cell subtypes or reflects broader lineage failure.

In [ ]:
confusion_tables = {}
for rep in REP_ORDER:
    pred_path = DONOR_CV_DIR / f"donor_cv_{rep}_all_predictions.csv"
    if not pred_path.exists():
        print("missing:", pred_path)
        continue
    fig, ax, cm = plot_confusion_matrix_from_predictions(
        pred_path,
        out_file=DIAG_FIGURES / f"blood_donor_cv_{rep}_confusion.png",
        title=f"Blood Atlas donor-held-out CV: {rep.upper()}",
    )
    plt.show()
    cm.to_csv(DIAG_RESULTS / f"blood_donor_cv_{rep}_row_normalized_confusion.csv")
    confusion_tables[rep] = cm

## Per-class F1 across representations

This is the compact companion to the confusion matrices. It makes the heterogeneous difficulty of the seven labels visible instead of hiding it inside macro-F1.

In [ ]:
f1_matrix = load_per_class_f1_matrix(DONOR_CV_DIR, rep_order=REP_ORDER)
display(f1_matrix)
fig, ax = plot_per_class_f1_heatmap(
    f1_matrix,
    rep_order=REP_ORDER,
    out_file=DIAG_FIGURES / "blood_donor_cv_per_class_f1_heatmap.png",
    title="Blood Atlas: per-class F1 under donor-held-out CV",
)
plt.show()

# 4. Broad-lineage marker sanity check

This is a QC diagnostic, not a manuscript endpoint. If broad lineage labels are correct, canonical lineage markers should still show recognizable enrichment even if fine T-cell subtype boundaries are hard.

The helper uses `adata.raw` when available so the check is not restricted to the 1,000 HVGs.

In [ ]:
marker_sets = {
    "T": ["CD3D", "CD3E", "TRAC"],
    "B": ["MS4A1", "CD79A", "CD74"],
    "Myeloid": ["LST1", "LILRB1", "CTSS", "LYZ"],
    "NK": ["NKG7", "GNLY", "PRF1"],
}
markers = [g for genes in marker_sets.values() for g in genes]

try:
    marker_means, missing_markers = marker_group_means(
        adata,
        markers,
        group_col=CELLTYPE_COL,
        use_raw=True,
    )
    marker_z = zscore_columns(marker_means)
    display(marker_means)
    print("Missing markers:", missing_markers)

    fig, ax = plt.subplots(figsize=(11, 5.5))
    im = ax.imshow(marker_z.values, aspect="auto", cmap="coolwarm", vmin=-2.5, vmax=2.5)
    ax.set_xticks(np.arange(marker_z.shape[1]))
    ax.set_xticklabels(marker_z.columns, rotation=60, ha="right")
    ax.set_yticks(np.arange(marker_z.shape[0]))
    ax.set_yticklabels(marker_z.index)
    ax.set_xlabel("Marker")
    ax.set_ylabel("Benchmark cell type")
    ax.set_title("Blood Atlas broad-lineage marker sanity check (gene-wise z scores)")
    fig.colorbar(im, ax=ax, label="z score across cell types")
    fig.tight_layout()
    fig.savefig(DIAG_FIGURES / "blood_marker_sanity_heatmap.png", dpi=300, bbox_inches="tight")
    plt.show()
except ValueError as exc:
    print("Marker check skipped:", exc)
    print("If var_names are Ensembl IDs, map symbols before using this section.")

# 5. Post-cap donor × cell-type composition

The question is not only whether the **global** benchmark is balanced. We want to know how much each donor's class composition differs from the cohort-wide composition.

We quantify:

- donor-specific proportions;
- normalized Shannon entropy;
- Jensen–Shannon divergence (JSD) from the global class distribution;
- per-cell-type SD/IQR of donor proportions.

In [ ]:
labels = sorted(adata.obs[CELLTYPE_COL].astype(str).unique())
post_counts = donor_celltype_counts(
    adata.obs,
    donor_col=DONOR_COL,
    celltype_col=CELLTYPE_COL,
    labels=labels,
)
post_props = donor_celltype_proportions(post_counts)
post_donor_metrics, post_celltype_metrics = summarize_donor_composition(post_counts)
post_counts.to_csv(DIAG_RESULTS / "composition/postcap_donor_celltype_counts.csv")
post_props.to_csv(DIAG_RESULTS / "composition/postcap_donor_celltype_proportions.csv")
post_donor_metrics.to_csv(DIAG_RESULTS / "composition/postcap_donor_metrics.csv", index=False)
post_celltype_metrics.to_csv(DIAG_RESULTS / "composition/postcap_celltype_metrics.csv", index=False)

display(post_celltype_metrics.sort_values("sd_proportion", ascending=False))
display(post_donor_metrics.describe())

In [ ]:
# Sort donors by JSD to make the composition heatmap easier to inspect.
donor_order = post_donor_metrics.sort_values("js_divergence_from_global")["donor"].tolist()
plot_props = post_props.reindex(donor_order)
fig, ax = plt.subplots(figsize=(9, 18))
im = ax.imshow(plot_props.values, aspect="auto", cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(np.arange(len(plot_props.columns)))
ax.set_xticklabels(plot_props.columns, rotation=60, ha="right")
ax.set_yticks([])
ax.set_xlabel("Cell type")
ax.set_ylabel("Donors (ordered by composition JSD)")
ax.set_title("Blood Atlas post-cap donor-specific cell-type proportions")
fig.colorbar(im, ax=ax, label="Within-donor proportion")
fig.tight_layout()
fig.savefig(DIAG_FIGURES / "blood_postcap_donor_composition_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

# 6. Did the donor × cell-type cap flatten the natural composition?

This analysis requires only the original metadata CSV, not the full expression matrix. It compares donor composition **before** Stage-0 sampling with the current capped benchmark, restricted to the same seven final labels.

In [ ]:
pre_counts = None
composition_design_summary = None
if FULL_METADATA.exists():
    meta = pd.read_csv(FULL_METADATA)
    donor_source = "donor_id" if "donor_id" in meta.columns else "Donor_id"
    celltype_source = "cell_type" if "cell_type" in meta.columns else "Cluster_names"
    if donor_source not in meta.columns or celltype_source not in meta.columns:
        raise KeyError("Could not identify donor/cell-type columns in the source metadata.")

    meta_work = meta[[donor_source, celltype_source]].dropna().copy()
    meta_work = meta_work.rename(columns={donor_source: DONOR_COL, celltype_source: CELLTYPE_COL})
    meta_work[DONOR_COL] = meta_work[DONOR_COL].astype(str)
    meta_work[CELLTYPE_COL] = meta_work[CELLTYPE_COL].astype(str)
    meta_work = meta_work[meta_work[CELLTYPE_COL].isin(labels)].copy()

    pre_counts = donor_celltype_counts(
        meta_work,
        donor_col=DONOR_COL,
        celltype_col=CELLTYPE_COL,
        labels=labels,
    )
    pre_props = donor_celltype_proportions(pre_counts)
    pre_donor_metrics, pre_celltype_metrics = summarize_donor_composition(pre_counts)

    pre_counts.to_csv(DIAG_RESULTS / "composition/precap_donor_celltype_counts.csv")
    pre_props.to_csv(DIAG_RESULTS / "composition/precap_donor_celltype_proportions.csv")
    pre_donor_metrics.to_csv(DIAG_RESULTS / "composition/precap_donor_metrics.csv", index=False)
    pre_celltype_metrics.to_csv(DIAG_RESULTS / "composition/precap_celltype_metrics.csv", index=False)

    composition_design_summary = pd.DataFrame([
        {"design": "pre-cap source metadata", **composition_summary_row(pre_counts)},
        {"design": "post-cap benchmark", **composition_summary_row(post_counts)},
    ])
    display(composition_design_summary)
    composition_design_summary.to_csv(
        DIAG_RESULTS / "composition/pre_vs_post_cap_summary.csv", index=False
    )
else:
    print("Source metadata unavailable; pre-vs-post cap composition comparison skipped.")

In [ ]:
if composition_design_summary is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
    axes[0].bar(
        composition_design_summary["design"],
        composition_design_summary["mean_js_divergence"],
        edgecolor="black",
    )
    axes[0].set_ylabel("Mean donor JSD from global composition")
    axes[0].set_title("Donor composition heterogeneity")
    axes[0].tick_params(axis="x", rotation=20)

    axes[1].bar(
        composition_design_summary["design"],
        composition_design_summary["mean_celltype_sd"],
        edgecolor="black",
    )
    axes[1].set_ylabel("Mean SD of cell-type proportion")
    axes[1].set_title("Across-donor proportion variability")
    axes[1].tick_params(axis="x", rotation=20)
    fig.tight_layout()
    fig.savefig(DIAG_FIGURES / "blood_pre_vs_post_cap_composition.png", dpi=300, bbox_inches="tight")
    plt.show()

# 7. Exploratory cross-dataset composition heterogeneity vs evaluation gap

This is intentionally descriptive: with only a handful of datasets, it is **not** a predictive model and we should not emphasize a p-value. The purpose is to see whether datasets with more donor-specific class composition also tend to show a larger random-cell minus donor-held-out gap.

The cell below skips any dataset whose checkpoint/results are absent.

In [ ]:
def _find_random_metrics(root: Path):
    candidates = [
        root / "random_split/random_split_repeated_metrics.csv",
        root / "random_split/metrics.csv",
    ]
    candidates += sorted((root / "random_split").glob("*repeated*metrics*.csv"))
    return next((p for p in candidates if p.exists()), None)


def _pca_gap(results_root: Path):
    random_path = _find_random_metrics(results_root)
    donor_path = results_root / "donor_cv/metrics.csv"
    if random_path is None or not donor_path.exists():
        return np.nan
    rnd = pd.read_csv(random_path)
    dnr = pd.read_csv(donor_path)
    rnd["representation"] = rnd["representation"].astype(str).replace({"X_pca": "pca"})
    dnr["representation"] = dnr["representation"].astype(str).replace({"X_pca": "pca"})
    r = rnd.loc[rnd["representation"].eq("pca"), "macro_f1"]
    d = dnr.loc[dnr["representation"].eq("pca"), "macro_f1_mean"]
    return float(r.mean() - d.mean()) if len(r) and len(d) else np.nan


dataset_specs = {
    "Blood Atlas": ("data/blood_atlas/blood_atlas_benchmark_ready.h5ad", "donor_id", "cell_type", "results/blood_atlas_no_batch"),
    "PBMC": ("data/PBMC_Stephenson/stephenson_benchmark_ready.h5ad", "patient_id", "cell_type", "results/pbmc_no_batch"),
    "Lung": ("data/lung/lung_benchmark_ready.h5ad", "donor_id", "cell_type", "results/lung_no_batch"),
    "Kidney": ("data/kidney/kidney_benchmark_ready.h5ad", "donor_id", "cell_type", "results/kidney_no_batch"),
    "Pancreas": ("data/pancreas/pancreas_benchmark_ready.h5ad", "donor_id", "cell_type", "results/pancreas_no_batch"),
}
rows = []
for name, (rel_path, donor_col, cell_col, result_rel) in dataset_specs.items():
    path = REPO_ROOT / rel_path
    if not path.exists():
        continue
    a = sc.read_h5ad(path, backed="r")
    try:
        if donor_col not in a.obs or cell_col not in a.obs:
            continue
        c = donor_celltype_counts(a.obs, donor_col=donor_col, celltype_col=cell_col)
        row = composition_summary_row(c, dataset=name)
        row["pca_random_minus_donor_f1"] = _pca_gap(REPO_ROOT / result_rel)
        rows.append(row)
    finally:
        try:
            a.file.close()
        except Exception:
            pass

cross_dataset = pd.DataFrame(rows)
display(cross_dataset)
if len(cross_dataset) >= 2:
    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.scatter(cross_dataset["mean_js_divergence"], cross_dataset["pca_random_minus_donor_f1"], s=55)
    for _, row in cross_dataset.iterrows():
        ax.annotate(row["dataset"], (row["mean_js_divergence"], row["pca_random_minus_donor_f1"]), xytext=(4, 4), textcoords="offset points")
    ax.axhline(0, linewidth=1, linestyle="--")
    ax.set_xlabel("Mean donor JSD from global cell-type composition")
    ax.set_ylabel("PCA macro-F1: random − donor-held-out")
    ax.set_title("Exploratory dataset-level composition vs evaluation gap")
    fig.tight_layout()
    fig.savefig(DIAG_FIGURES / "cross_dataset_composition_vs_gap.png", dpi=300, bbox_inches="tight")
    plt.show()

# 8. Repeated 20-donor cohorts

The existing donor-ablation curve asks how performance changes with the **number of training donors inside the full cohort**. This analysis asks a different question: *what would the entire random-vs-donor comparison look like if Blood Atlas itself contained only 20 donors?*

We repeatedly sample 20 total donors and rerun both evaluation schemes. This isolates whether the current 166-donor diversity is masking an evaluation gap.

In [ ]:
def run_sensitivity_cli(*args):
    cmd = [sys.executable, str(REPO_ROOT / "scripts/run_blood_atlas_sensitivity.py"), *map(str, args)]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

DONOR20_RAW = DIAG_RESULTS / "donor20/donor20_random_vs_donor_raw.csv"
if RUN_DONOR20:
    run_sensitivity_cli("--donor20", "--n-donors", 20, "--n-cohorts", 10)

if DONOR20_RAW.exists():
    donor20 = pd.read_csv(DONOR20_RAW)
    display(donor20.groupby("representation")[["random_macro_f1", "donor_macro_f1_mean", "delta_macro_f1_random_minus_donor"]].agg(["mean", "std"]))

    fig, ax = plt.subplots(figsize=(8, 5))
    reps = [r for r in REP_ORDER if r in set(donor20["representation"])]
    positions = np.arange(len(reps))
    for i, rep in enumerate(reps):
        vals = donor20.loc[donor20["representation"].eq(rep), "delta_macro_f1_random_minus_donor"]
        x = np.full(len(vals), i) + np.linspace(-0.08, 0.08, len(vals))
        ax.scatter(x, vals, alpha=0.75)
        ax.hlines(vals.mean(), i - 0.22, i + 0.22, linewidth=2)
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_xticks(positions)
    ax.set_xticklabels([r.upper() if r != "harmony" else "Harmony" for r in reps])
    ax.set_ylabel("Macro-F1: random − donor-held-out")
    ax.set_title("Blood Atlas: evaluation gap across repeated 20-donor cohorts")
    fig.tight_layout()
    fig.savefig(DIAG_FIGURES / "blood_donor20_gap_distribution.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("20-donor outputs not present. Set RUN_DONOR20=True once to generate them.")

## Nested donor ablation inside the 20-donor cohorts

This is deliberately modest (`k = 5, 10, 15`) rather than recreating the entire full-cohort ablation grid. It checks whether donor-held-out performance has a strong donor-number learning curve within reduced cohorts.

In [ ]:
DONOR20_ABLATION = DIAG_RESULTS / "donor20_ablation/donor20_ablation_summary.csv"
if RUN_DONOR20_ABLATION:
    run_sensitivity_cli("--donor20-ablation", "--ablation-k", 5, 10, 15)

if DONOR20_ABLATION.exists():
    abl = pd.read_csv(DONOR20_ABLATION)
    display(abl)
    fig, ax = plt.subplots(figsize=(7.5, 5))
    for rep in REP_ORDER:
        sub = abl[abl["representation"].eq(rep)].sort_values("k_train_donors")
        if sub.empty:
            continue
        ax.errorbar(sub["k_train_donors"], sub["macro_f1_mean"], yerr=sub["macro_f1_std"], marker="o", capsize=4, label=rep)
    ax.set_xlabel("Training donors within 20-donor cohorts")
    ax.set_ylabel("Donor-held-out macro F1")
    ax.set_title("Blood Atlas: nested donor ablation in reduced cohorts")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(DIAG_FIGURES / "blood_donor20_nested_ablation.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Nested-ablation output not present. Set RUN_DONOR20_ABLATION=True once to generate it.")

# 9. Null controls: global vs within-donor label permutation

These two controls answer different questions.

- **Global permutation** destroys expression→cell-type structure and donor-specific class priors.
- **Within-donor permutation** destroys expression→cell-type structure while preserving each donor's class proportions.

If random splitting retains an advantage only under the within-donor null, donor-specific expression signatures plus donor-specific class priors can create apparent predictive signal even without real cell-type separability. On the current donor × cell-type-capped Blood benchmark, we expect that mechanism to be attenuated if the cap made donor proportions unusually uniform.

In [ ]:
PERM_RAW = DIAG_RESULTS / "label_permutation/label_permutation_raw.csv"
if RUN_PERMUTATIONS:
    run_sensitivity_cli("--permutations", "--n-permutations", 10)

if PERM_RAW.exists():
    perm = pd.read_csv(PERM_RAW)
    display(perm.groupby(["permutation_mode", "representation"])[["random_macro_f1", "donor_macro_f1_mean", "delta_macro_f1_random_minus_donor"]].agg(["mean", "std"]))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
    for ax, mode in zip(axes, ["global", "within_donor"]):
        sub = perm[perm["permutation_mode"].eq(mode)]
        reps = [r for r in REP_ORDER if r in set(sub["representation"])]
        data = [sub.loc[sub["representation"].eq(r), "delta_macro_f1_random_minus_donor"].values for r in reps]
        ax.boxplot(data, tick_labels=reps)
        ax.axhline(0, linestyle="--", linewidth=1)
        ax.set_title(mode.replace("_", " ").title())
        ax.set_xlabel("Representation")
    axes[0].set_ylabel("Macro-F1: random − donor-held-out")
    fig.suptitle("Blood Atlas null controls")
    fig.tight_layout()
    fig.savefig(DIAG_FIGURES / "blood_label_permutation_gaps.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Permutation outputs not present. Set RUN_PERMUTATIONS=True once to generate them.")

# 10. Four-class diagnostic: collapse neighboring T-lineage labels

This is **not** a proposed replacement benchmark. It asks whether the low seven-class F1 is largely a label-granularity issue. The diagnostic maps CD4, CD8, γδ T and MAIT classes into a single `T cells` class while retaining NK, B and Myeloid separately.

In [ ]:
COARSE_RAW = DIAG_RESULTS / "coarse_labels/coarse_4class_raw.csv"
if RUN_COARSE_LABELS:
    run_sensitivity_cli("--coarse", "--coarse-repeats", 5)

if COARSE_RAW.exists():
    coarse = pd.read_csv(COARSE_RAW)
    display(coarse.groupby("representation")[["random_macro_f1", "donor_macro_f1_mean", "delta_macro_f1_random_minus_donor"]].agg(["mean", "std"]))

    # Compare against the original seven-class saved summaries without rerunning.
    random_candidates = [
        RESULTS_DIR / "random_split/random_split_repeated_metrics.csv",
        RESULTS_DIR / "random_split/metrics.csv",
    ] + sorted((RESULTS_DIR / "random_split").glob("*repeated*metrics*.csv"))
    random_path = next((p for p in random_candidates if p.exists()), None)
    donor_path = DONOR_CV_DIR / "metrics.csv"
    if random_path is not None and donor_path.exists():
        rnd = pd.read_csv(random_path)
        dnr = pd.read_csv(donor_path)
        orig = []
        for rep in REP_ORDER:
            rv = rnd.loc[rnd["representation"].astype(str).eq(rep), "macro_f1"]
            dv = dnr.loc[dnr["representation"].astype(str).eq(rep), "macro_f1_mean"]
            if len(rv) and len(dv):
                orig.append({"representation": rep, "seven_class_random": rv.mean(), "seven_class_donor": dv.mean()})
        orig = pd.DataFrame(orig)
        four = coarse.groupby("representation", as_index=False).agg(four_class_random=("random_macro_f1", "mean"), four_class_donor=("donor_macro_f1_mean", "mean"))
        display(orig.merge(four, on="representation", how="outer"))
else:
    print("Coarse-label output not present. Set RUN_COARSE_LABELS=True once to generate it.")

# 11. Composition-preserving alternative cohort

The primary Stage-0 design caps every donor × cell-type combination at 100 cells. That is useful for controlling sample size, but it may suppress donor-specific class-prior heterogeneity.

The alternative sensitivity cohort therefore:

1. verifies exact source metadata ↔ expression barcode identity;
2. restricts to the same seven benchmark labels;
3. caps **only total cells per donor** (default 650), preserving each donor's natural within-donor composition;
4. runs the same Blood preprocessing (HVG/PCA/Harmony; no scVI; no re-normalization of transformed `X`);
5. compares random vs donor-held-out gaps with the primary cohort.

This is a sensitivity cohort, **not** a replacement for the primary benchmark.

In [ ]:
ALT_STAGE0 = REPO_ROOT / "data/blood_atlas/blood_atlas_donor_only_subsampled.h5ad"
ALT_READY = REPO_ROOT / "data/blood_atlas/blood_atlas_composition_preserving_benchmark_ready.h5ad"

if RUN_COMPOSITION_STAGE0:
    if not (FULL_H5AD.exists() and FULL_METADATA.exists()):
        raise FileNotFoundError("Set valid FULL_H5AD and FULL_METADATA first.")
    run_sensitivity_cli(
        "--prepare-composition-stage0",
        "--full-h5ad", FULL_H5AD,
        "--metadata-csv", FULL_METADATA,
        "--metadata-id-col", METADATA_ID_COL,
        "--max-cells-per-donor", 650,
    )

if RUN_COMPOSITION_PREPROCESSING:
    subprocess.run(
        [
            sys.executable,
            str(REPO_ROOT / "scripts/preprocess.py"),
            "--config",
            str(REPO_ROOT / "configs/preprocessing/blood_atlas_composition_preserving.yaml"),
        ],
        cwd=REPO_ROOT,
        check=True,
    )

print("Alternative Stage-0 exists:", ALT_STAGE0.exists())
print("Alternative benchmark-ready object exists:", ALT_READY.exists())

In [ ]:
COMP_RAW = DIAG_RESULTS / "composition_preserving/random_vs_donor_raw.csv"
if RUN_COMPOSITION_EVAL:
    run_sensitivity_cli("--composition-eval", "--composition-repeats", 5)

if COMP_RAW.exists():
    comp = pd.read_csv(COMP_RAW)
    display(comp.groupby(["cohort_design", "representation"])[["random_macro_f1", "donor_macro_f1_mean", "delta_macro_f1_random_minus_donor"]].agg(["mean", "std"]))

    fig, ax = plt.subplots(figsize=(8, 5))
    summary = comp.groupby(["cohort_design", "representation"], as_index=False).agg(
        gap=("delta_macro_f1_random_minus_donor", "mean"),
        gap_sd=("delta_macro_f1_random_minus_donor", "std"),
    )
    reps = [r for r in REP_ORDER if r in set(summary["representation"])]
    x = np.arange(len(reps))
    width = 0.36
    designs = ["primary_capped", "composition_preserving"]
    for j, design in enumerate(designs):
        sub = summary[summary["cohort_design"].eq(design)].set_index("representation").reindex(reps)
        ax.bar(x + (j - 0.5) * width, sub["gap"], width, yerr=sub["gap_sd"], capsize=4, edgecolor="black", label=design.replace("_", " "))
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(reps)
    ax.set_ylabel("Macro-F1: random − donor-held-out")
    ax.set_title("Does preserving natural donor composition increase the evaluation gap?")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(DIAG_FIGURES / "blood_composition_preserving_gap_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Composition-preserving evaluation not present. Generate/preprocess ALT cohort, then set RUN_COMPOSITION_EVAL=True.")

## Strong mechanistic control: within-donor permutation on the composition-preserving cohort

If donor-specific composition is a leakage mechanism, the most revealing null is the **within-donor label permutation on the composition-preserving cohort**. It leaves donor class priors intact while removing true expression→cell-type biology.

In [ ]:
COMP_PERM = DIAG_RESULTS / "composition_preserving/within_donor_permutation_raw.csv"
if RUN_COMPOSITION_PERMUTATION:
    run_sensitivity_cli("--composition-permutation", "--n-permutations", 10)

if COMP_PERM.exists():
    comp_perm = pd.read_csv(COMP_PERM)
    display(comp_perm.groupby("representation")[["random_macro_f1", "donor_macro_f1_mean", "delta_macro_f1_random_minus_donor"]].agg(["mean", "std"]))
else:
    print("Composition-preserving permutation control not present.")

# 12. Interpretation checklist

Use the completed notebook to decide among several non-exclusive explanations rather than forcing a single story:

- **Source alignment problem:** if the barcode audit fails, fix/rebuild Blood Atlas before interpreting any of the old Blood results.
- **Label-granularity floor:** if broad markers look coherent and the four-class task improves sharply, much of the low F1 comes from difficult neighboring immune subtypes.
- **Composition flattening:** if pre-cap donor composition is substantially more heterogeneous than post-cap, the current Stage-0 design removed one potential random-split advantage.
- **Large-donor-cohort robustness:** if repeated 20-donor subsets show a larger random−donor gap than the 166-donor cohort, donor diversity itself is part of why the full Blood result is insensitive to split strategy.
- **Donor-prior leakage mechanism:** compare global vs within-donor permutations, especially on the composition-preserving cohort.
- **Persistent near-zero gap:** if the gap remains small after all of the above while real-label performance remains above the permutation null, Blood Atlas is a useful real-data boundary case where predictive signal exists but evaluation-scheme inflation is weak.

Once these questions are resolved, stop probing Blood Atlas and return to manuscript figure assembly.